In [1]:
# Features engineering
import os
import pyedflib
import pandas as pd
import numpy as np
import time 
import re

### récupére les données :

- EEG SD ACC X,EEG SD ACC Y,EEG SD ACC Z,ECGEMG SD ACC X,ECGEMG SD ACC Y,ECGEMG SD ACC Z

- Ne convertir que les patients figurant dans la liste des patients retenus (/home/aura-fabien/SizeIT/Datas/Preprocessing_datas/bdd_seizure_modele.csv)

- La taille des fichiers est réduite en ne conservant que 5 chiffres après la virgule dans les csv. 

In [12]:
import os
import glob

folder = "/home/aura-fabien/SizeIT/Datas/Csv_acc"

files = glob.glob(os.path.join(folder, "*"))

for f in files:
    if os.path.isfile(f):
        os.remove(f)

print("Dossier vidé ✅")

Dossier vidé ✅


In [ ]:
dir_all_pat = '/data2/datasets/seizeit2-dataset/ds005873-1.1.0/'
folder_patients = dir_all_pat
output_folders = '/home/aura-fabien/SizeIT/Datas/Csv_acc'

bdd_modele = pd.read_csv('/home/aura-fabien/SizeIT/Datas/Preprocessing_datas/bdd_seizure_modele.csv')
l_patient_modele = bdd_modele.patient.unique()


list_all_acc_csv = []
for root, dirs, files in os.walk(dir_all_pat):
    for file in files:
        if file.lower().endswith("mov.edf"):
            edf_path = os.path.join(root, file)
            list_all_acc_csv.append(edf_path)

start_time = time.time()

# --- Dossiers ---
input_folder = list_all_acc_csv
output_folder = "/home/aura-fabien/SizeIT/Datas/Csv_acc"
print(f'{len(input_folder)} fichiers à convertir')

# os.makedirs(output_folder, exist_ok=True)

# --- Conversion ---
for no_file,edf_path in enumerate(input_folder):
    
    print(edf_path)
    
    match = re.search(r"sub-(\d{3})", edf_path)
    patient_id = int(match.group(1))
    
    if patient_id in l_patient_modele:

        print(f"Conversion de : {edf_path}")

        # Lecture EDF
        f = pyedflib.EdfReader(edf_path)
        n_signals = f.signals_in_file
        signal_labels = f.getSignalLabels()

        acc_indices = [
            i for i, label in enumerate(signal_labels)
            if "ACC" in label.upper()
        ]
        
        if not acc_indices:
            print("  → Aucun canal ACC trouvé")
            continue
        
        data = {}
                
        for i in acc_indices:
                    data[signal_labels[i]] = f.readSignal(i)
            
        # DataFrame robuste (si longueurs différentes)
        df = pd.DataFrame(
            {k: pd.Series(v) for k, v in data.items()}
        )
        
        start_datetime = f.getStartdatetime()  # datetime naïf EXACT du fichier

        fs = f.getSampleFrequency(acc_indices[0])
        n_samples = len(df)

        time_offsets = pd.to_timedelta(np.arange(n_samples) / fs, unit="s")

        df.insert(0, "time", pd.Timestamp(start_datetime) + time_offsets)

        f.close()
        
        csv_name = edf_path.replace(".edf", ".csv")
        csv_name = csv_name.split("/")[-1]
        csv_path = output_folder + '/' + csv_name
        df.to_csv(csv_path, index=False,float_format="%.5f")
        
end_time = time.time()
print('temps ecoulé = ', end_time - start_time)



2551 fichiers à convertir
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-01_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-02_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-03_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-04_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-05_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-06_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-049/ses-01/mov/sub-049_ses-01_task-szMonitoring_run-07_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-051/ses-01/mov/sub-051_ses-01_task-szMonitoring_run-01_mov.edf
/data2/datasets/seizeit2-dataset/ds005873-1.1.0/sub-05

In [4]:
df.head(1)

NameError: name 'df' is not defined